# FleetIQ / SmartTaxi — 06. Baseline Modeling

## Objective
Fit and evaluate the Historical Average Speed heuristic baseline model (`src.models.baseline`).
Measures global urban speed and hourly segmented speeds to establish the lower bound benchmark.


In [ ]:
import sys
sys.path.append('..')

import pandas as pd
from src.models.baseline import HistoricalAvgSpeedBaseline
from src.evaluation.metrics import evaluate_all

train_df = pd.read_parquet('../data/processed/train.parquet')
test_df = pd.read_parquet('../data/processed/test.parquet')

baseline = HistoricalAvgSpeedBaseline(default_speed_kmh=25.0)
baseline.fit(
    distance_km=train_df['distance_haversine_km'],
    duration_min=train_df['trip_duration_minutes'],
    hours=train_df['pickup_hour'],
)

print(f'Fitted Global Speed: {baseline.global_speed_kmh:.2f} km/h')
print('Sample Hourly Speeds (km/h):', {h: round(baseline.hourly_speeds[h], 1) for h in range(0, 24, 4)})


## 1. Evaluate Baseline on Held-out Out-of-Time Test Set


In [ ]:
y_test = test_df['trip_duration_minutes'].values
preds = baseline.predict(test_df['distance_haversine_km'], hours=test_df['pickup_hour'])

metrics = evaluate_all(y_test, preds)
for k, v in metrics.items():
    print(f'{k:<16}: {v}')
